# Lesson 8 : Human-in-the-loop (HITL)

The participant in agent is not only a computer agent or a program code, but also a human can be involved in the conversation (agentic loop).

Using Agent Framework, you can apply either of the following approaches to implement human involving - such as, approval request.

- You can handle human input in local tool's calling. Especially in Agent Framework, you can specify ```@ai_function(approval_mode="always_require")``` decorator in tool's function definition, and you can process the human-in-the-loop separately from the main logic. Please see [official document](https://learn.microsoft.com/en-us/agent-framework/tutorials/agents/function-tools-approvals?pivots=programming-language-python) for this implementation.
- In workflows, you can invoke ```request_info()``` method in workflow context and you can handle human input as workflow's event outside of your workflow. This pattern can integrate with checkpointing in workflows (see [Lesson 7](./07_workflow.ipynb) for workflow's chekpoint capability), and you can then handle human-in-the-loop in long-running (2-3 days or more) process.

In this exercise, using the latter approach, we briefly explore the implementation of human-in-the-loop.

> Note : This example uses generic ```WorkflowBuilder```, but in high-level workflow builders (```SequentialBuilder```, ```ConcurrentBuilder```, ```GroupChatBuilder```, or ```HandoffBuilder```), you can use ```with_request_info()``` instead. (This ```with_request_info()``` method calls ```request_info()``` internally.)

First we initialize the client.

In [1]:
from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential

credential = AzureCliCredential()
client = AzureAIClient(credential=credential)

Now we build a workflow with human interaction as follows.

In this example, I have changed the example in Lesson 7 (travel planning example) to include human approval at the end.

I have only replaced ```ResponseGenerator``` executor in Lesson 7 with new ```HumanReview``` executor.  
In this executor, ```request_info()``` (see above) is invoked in handler method.  
After the human performs approval response, the method with ```@response_handler``` decorator will be invoked.

This example briefly show the approval result in text, but you can also build revision loop by building a loop in the workflow's edge and invoking ```ctx.send_message()``` in response handler. (If you have time, please try this implementation.)

In [2]:
from dataclasses import dataclass
from agent_framework import ChatAgent, ChatMessage
from agent_framework import (
    WorkflowBuilder,
    Executor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    WorkflowContext,
    handler,
    response_handler
)

@dataclass
class ApprovalRequest:
    # this class is serialized when using checkpoint
    prompt: str = ""
    draft: str = ""

class HumanReview(Executor):
    @handler
    async def generate_human_review_request(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext
    ) -> None:
        await ctx.request_info(
            request_data=ApprovalRequest(
                prompt=(
                    "Review the draft and reply 'approve' or 'reject' in text."  # "旅行計画のドラフトを確認し、'approve' または 'reject' のテキストで回答してください。"
                    "(Otherwise this draft will be rejected.)"  # "(それ以外が入力された場合、処理は却下されます。)"
                ),
                draft=response.agent_run_response.text,
            ),
            response_type=str,
        )

    @response_handler
    async def do_after_human_response(
        self,
        original_request: ApprovalRequest,
        approve_or_reject: str,
        ctx: WorkflowContext[AgentExecutorRequest | str, str],
    ) -> None:
        if approve_or_reject.strip().lower() == "approve":
            await ctx.yield_output(f"Approved ! :\n----------\n {original_request.draft}")
        else:
            await ctx.yield_output("The draft is rejected.")

workflow = (
    WorkflowBuilder()
    .register_agent(
        lambda: ChatAgent(
            name="PlanningAgent",
            chat_client=client,
            instructions="Your task is to help users plan and concisely summarize it in five bullet points.",  # "ユーザーの計画作成を支援し、箇条書きの 5 項目に簡潔にまとめます。"
        ),
        name="PlanningAgent",
        output_response=True,
    )
    .register_agent(
        lambda: ChatAgent(
            name="ReviseAgent",
            chat_client=client,
            instructions="Your task is to review the plan you receive and to refine it further.",  # "与えられた計画を確認して、より洗練させます。"
        ),
        name="ReviseAgent",
        output_response=False,
    )
    .register_executor(
        lambda: HumanReview(id="final_human_review"),
        name="FinalReview"
    )
    .add_edge("PlanningAgent", "ReviseAgent")
    .add_edge("ReviseAgent", "FinalReview")
    .set_start_executor("PlanningAgent")
    .build()
)

Now we run this workflow.

When ```request_info()``` is invoked, ```RequestInfoEvent``` is captured in streaming event loop.   
Before proceeding to the next, the event loop is then suspended. (The workflow goes into idle state.)

In this code, the human approval response is temporarily stored in ```responses``` variable. (This code always return "approve", because Jupyter notebook cannot handle interactive response by humans.)

In [3]:
from agent_framework import RequestInfoEvent, WorkflowOutputEvent

responses: dict[str, str] = {}

async for event in workflow.run_stream("Create a plan for my travel in Osaka."):  # "大阪の旅行計画を作成してください。"
    if isinstance(event, RequestInfoEvent):
        approval_request = event.data
        request_id = event.request_id

        print("----------------------------------")
        print(approval_request.prompt)
        print("----------------------------------")
        print(approval_request.draft)
        response = "approve"   # always send approve

        responses[request_id] = response

----------------------------------
Review the draft and reply 'approve' or 'reject' in text.(Otherwise this draft will be rejected.)
----------------------------------
- **Lock logistics:** Choose dates/length, stay in **Namba** (food/nightlife) or **Umeda** (transit), and use **ICOCA** + Osaka Metro; consider the **Osaka Amazing Pass** if you’ll do multiple attractions in 1–2 days.  
- **Day 1 – Icons + night scene:** **Osaka Castle Park → Umeda Sky Building → Dotonbori/Shinsaibashi** for evening canal walk, lights, and street food.  
- **Day 2 – Food + old Osaka:** **Kuromon Ichiba** brunch → **Den Den Town** → **Shinsekai/Tsutenkaku**; focus on **takoyaki/okonomiyaki/kushikatsu**.  
- **Day 3 – Big-ticket day:** Do **USJ** (full day; reserve timed entry/Express Pass if needed) *or* **Kaiyukan Aquarium + Tempozan** for a lighter pace.  
- **Add 1–2 day trips:** Choose **Kyoto**, **Nara**, **Kobe**, or **Himeji**, returning to Osaka for dinner and nightlife.


Again start the workflow by responding approval request with ```send_responses_streaming()``` method as follows.  
The rest of process will be performed in workflow, and the workflow will then be completed.

After completion, you can see that the text "Approved !" is appended in the final output.

In [4]:
from agent_framework import WorkflowOutputEvent

async for event in workflow.send_responses_streaming(responses):
    if isinstance(event, WorkflowOutputEvent):
        final_output = event.data

print(final_output)

Approved ! :
----------
 - **Lock logistics:** Choose dates/length, stay in **Namba** (food/nightlife) or **Umeda** (transit), and use **ICOCA** + Osaka Metro; consider the **Osaka Amazing Pass** if you’ll do multiple attractions in 1–2 days.  
- **Day 1 – Icons + night scene:** **Osaka Castle Park → Umeda Sky Building → Dotonbori/Shinsaibashi** for evening canal walk, lights, and street food.  
- **Day 2 – Food + old Osaka:** **Kuromon Ichiba** brunch → **Den Den Town** → **Shinsekai/Tsutenkaku**; focus on **takoyaki/okonomiyaki/kushikatsu**.  
- **Day 3 – Big-ticket day:** Do **USJ** (full day; reserve timed entry/Express Pass if needed) *or* **Kaiyukan Aquarium + Tempozan** for a lighter pace.  
- **Add 1–2 day trips:** Choose **Kyoto**, **Nara**, **Kobe**, or **Himeji**, returning to Osaka for dinner and nightlife.
